In [1]:
# Generate Dataset
import numpy as np

np.random.seed(42)

X = np.random.uniform(-2, 2, (400, 3))
y = (
    np.sin(X[:, 0]) +
    0.5 * (X[:, 1] ** 2) -
    0.8 * X[:, 2]
)
y = y.reshape(-1, 1)

X = X.T
y = y.T
N = X.shape[1]

In [2]:
# Activation Functions

def relu(z):
    return np.maximum(0, z)

def drelu(z):
    return (z > 0).astype(float)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def dsigmoid(z):
    s = sigmoid(z)
    return s * (1 - s)

def tanh(z):
    return np.tanh(z)

def dtanh(z):
    return 1 - np.tanh(z)**2

def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha*z)

def dleaky_relu(z, alpha=0.01):
    return np.where(z > 0, 1, alpha)

def softplus(z):
    return np.log(1 + np.exp(z))

def dsoftplus(z):
    return sigmoid(z)

In [3]:
# Initialize Parameters
def initialize(layers):
    params = {}
    for i in range(1, len(layers)):
        params["W"+str(i)] = np.random.uniform(-0.5, 0.5, (layers[i], layers[i-1]))
        params["b"+str(i)] = np.zeros((layers[i], 1))
    return params

In [4]:
# Forward Propagation
def forward(X, params, activation):
    cache = {"A0": X}
    L = len(params) // 2

    for l in range(1, L+1):
        W = params["W"+str(l)]
        b = params["b"+str(l)]
        A_prev = cache["A"+str(l-1)]

        Z = W @ A_prev + b

        if l == L:
            A = Z   # Output layer linear (regression)
        else:
            A = activation(Z)

        cache["Z"+str(l)] = Z
        cache["A"+str(l)] = A

    return cache

In [6]:
# Backpropagation
def backward(params, cache, y, activation_deriv):
    grads = {}
    L = len(params) // 2

    A_final = cache["A"+str(L)]
    dA = (2/N) * (A_final - y)

    for l in reversed(range(1, L+1)):
        Z = cache["Z"+str(l)]
        A_prev = cache["A"+str(l-1)]

        if l == L:
            dZ = dA
        else:
            dZ = dA * activation_deriv(Z)

        grads["dW"+str(l)] = (1/N) * (dZ @ A_prev.T)
        grads["db"+str(l)] = (1/N) * np.sum(dZ, axis=1, keepdims=True)

        if l > 1:
            W = params["W"+str(l)]
            dA = W.T @ dZ

    return grads

In [7]:
# Update
def update(params, grads, lr):
    L = len(params)//2
    for l in range(1, L+1):
        params["W"+str(l)] -= lr * grads["dW"+str(l)]
        params["b"+str(l)] -= lr * grads["db"+str(l)]
    return params

In [8]:
# Train Model
def train(layers, activation, activation_deriv, epochs=1000, lr=0.01):

    params = initialize(layers)

    loss_200 = None

    for epoch in range(epochs):

        cache = forward(X, params, activation)
        y_hat = cache["A"+str(len(layers)-1)]

        loss = np.mean((y_hat - y)**2)

        if epoch == 199:
            loss_200 = loss

        grads = backward(params, cache, y, activation_deriv)
        params = update(params, grads, lr)

    # Gradient Norms (Final epoch)
    grad_l1 = np.linalg.norm(grads["dW1"])

    # Last hidden layer
    last_hidden = len(layers) - 2
    grad_last = np.linalg.norm(grads["dW"+str(last_hidden)])

    return loss, loss_200, grad_l1, grad_last

In [9]:
# Architectures
model_A = [3, 4, 1]
model_B = [3, 6, 6, 1]
model_C = [3, 8, 8, 8, 8, 1]
model_D = [3, 8, 8, 8, 8, 8, 8, 8, 8, 1]

In [10]:
print("MODEL A - ReLU")
print(train(model_A, relu, drelu))

print("MODEL B - ReLU")
print(train(model_B, relu, drelu))

print("MODEL C - ReLU")
print(train(model_C, relu, drelu))

print("MODEL D - ReLU")
print(train(model_D, relu, drelu))

print("MODEL A - Sigmoid")
print(train(model_A, sigmoid, dsigmoid))

print("MODEL B - Sigmoid")
print(train(model_B, sigmoid, dsigmoid))

print("MODEL C - Sigmoid")
print(train(model_C, sigmoid, dsigmoid))

print("MODEL D - Sigmoid")
print(train(model_D, sigmoid, dsigmoid))

MODEL A - ReLU
(np.float64(2.954449529102792), np.float64(3.1620077617906843), np.float64(0.002818328105617886), np.float64(0.002818328105617886))
MODEL B - ReLU
(np.float64(2.0582392351908454), np.float64(2.1165804313027814), np.float64(0.0015319871861991558), np.float64(0.0013155438011370085))
MODEL C - ReLU
(np.float64(2.3655860412248955), np.float64(2.42568668771064), np.float64(0.00010805129388447927), np.float64(0.0004397120028136236))
MODEL D - ReLU
(np.float64(2.3572306560797656), np.float64(2.4309639593120176), np.float64(1.8033836630896833e-05), np.float64(3.86886291578942e-05))
MODEL A - Sigmoid
(np.float64(2.2153898054074954), np.float64(2.3181140649423915), np.float64(0.0007981867644117074), np.float64(0.0007981867644117074))
MODEL B - Sigmoid
(np.float64(2.031344959759353), np.float64(2.0986644388958773), np.float64(0.0001819934983453095), np.float64(0.0007103362002404743))
MODEL C - Sigmoid
(np.float64(1.9690055988843491), np.float64(2.037585234290435), np.float64(3.7256